# **MUD Task 3: Drug-Drug Interaction using Deep Learning**

**Authors:** Laia Jané and Elisa Müller



## Table of Contents

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Define paths
Define the paths to the data and utils in your Drive unit:

In [ ]:
# Elisa
utilsdir='/content/drive/MyDrive/LAB MUD/Task 3/07-DDI-nn'
evaluatordir='/content/drive/MyDrive/LAB MUD/Task 3/lab_resources/DDI/util'
trainfile='/content/drive/MyDrive/LAB MUD/Task 3/07-DDI-nn/train.pck'
validationfile='/content/drive/MyDrive/LAB MUD/Task 3/07-DDI-nn/devel.pck'
testfile='/content/drive/MyDrive/LAB MUD/Task 3/07-DDI-nn/test.pck'
validationdir='/content/drive/MyDrive/LAB MUD/Task 3/lab_resources/DDI/data/devel'
testdir='/content/drive/MyDrive/LAB MUD/Task 3/lab_resources/DDI/data/test'
modelname ='model.keras'
outfile ='out.txt'

In [ ]:
# # Laia
# utilsdir='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/07-DDI-nn'
# evaluatordir='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/lab_resources/DDI/util'
# trainfile='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/07-DDI-nn/train.pck'
# validationfile='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/07-DDI-nn/devel.pck'
# testfile='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/07-DDI-nn/test.pck'
# validationdir='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/lab_resources/DDI/data/devel'
# testdir='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/lab_resources/DDI/data/test'
# modelname ='model.keras'
# outfile ='out.txt'

## 3. Set random seed

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

SEED = 123
os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

## 4. Add local paths and import modules

In [ ]:
!pip install -q transformers==4.33.2 accelerate
import sys
sys.path.insert(1,utilsdir) # Path to the utils folder on your Google Drive disk
sys.path.insert(1,evaluatordir) # Path to the evaluator folder on your Google Drive disk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.9/119.9 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 22.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.8 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [ ]:
from contextlib import redirect_stdout

import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification

from codemaps import *
from dataset import *

import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
FEATURES = ["input_ids", "attention_mask"]

## 5. Load train and validation data

In [ ]:
# load train and validation data
traindata = Dataset(trainfile)
valdata = Dataset(validationfile)

# create indexes from training data
max_len = 150
suf_len = 5
codes = Codemaps(traindata, max_len)

# encode datasets
Xt = codes.encode_words(traindata, features=FEATURES)
Yt = codes.encode_labels(traindata)
Xv = codes.encode_words(valdata, features=FEATURES)
Yv = codes.encode_labels(valdata)

n_tags = codes.get_n_labels()
max_len = codes.maxlen

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 4. Define `build_network`

In [ ]:
def build_model(n_labels):
    model = AutoModelForSequenceClassification.from_pretrained(
        'dmis-lab/biobert-base-cased-v1.2',
        num_labels=n_labels
    )
    return model

## 6. Create the model

In [ ]:
model = build_model(codes.get_n_labels())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print('Model loaded on', device)
print(model.config)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda
BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_si

## 7. Train the model

In [ ]:
from tqdm.auto import tqdm

train_labels = torch.tensor(np.argmax(Yt, axis=1), dtype=torch.long)
val_labels = torch.tensor(np.argmax(Yv, axis=1), dtype=torch.long)

train_dataset = TensorDataset(
    torch.tensor(Xt[0], dtype=torch.long),
    torch.tensor(Xt[1], dtype=torch.long),
    train_labels
)

val_dataset = TensorDataset(
    torch.tensor(Xv[0], dtype=torch.long),
    torch.tensor(Xv[1], dtype=torch.long),
    val_labels
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

print("Device:", device)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

EPOCHS = 3

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    model.train()
    train_loss = 0.0

    for step, batch in enumerate(tqdm(train_loader, desc="Training")):
        input_ids_batch, attention_mask_batch, labels_batch = [t.to(device) for t in batch]

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids_batch,
            attention_mask=attention_mask_batch,
            labels=labels_batch
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    print(f"Epoch {epoch+1} train loss: {train_loss:.4f}")

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            input_ids_batch, attention_mask_batch, labels_batch = [t.to(device) for t in batch]

            outputs = model(
                input_ids=input_ids_batch,
                attention_mask=attention_mask_batch,
                labels=labels_batch
            )

            val_loss += outputs.loss.item()

            preds = outputs.logits.argmax(dim=-1)
            correct += (preds == labels_batch).sum().item()
            total += labels_batch.size(0)

    val_loss /= len(val_loader)
    print(f"Epoch {epoch+1} val loss: {val_loss:.4f}, val acc: {correct/total:.4f}")

model.save_pretrained("bert_model")
codes.save("bert_model")

Device: cuda
Training batches: 2894
Validation batches: 289

Epoch 1/3


Training:   0%|          | 0/2894 [00:00<?, ?it/s]

Epoch 1 train loss: 0.1261


Validation:   0%|          | 0/289 [00:00<?, ?it/s]

Epoch 1 val loss: 0.2287, val acc: 0.9246

Epoch 2/3


...

Training:   0%|          | 0/2894 [00:00<?, ?it/s]

Epoch 3 train loss: 0.0584


Validation:   0%|          | 0/289 [00:00<?, ?it/s]

Epoch 3 val loss: 0.2481, val acc: 0.9404


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# 8. Predict

In [ ]:
#import sys
import evaluator

In [ ]:
def output_interactions(data, preds, outfile) :

   #print(testdata[0])
   outf = open(outfile, 'w')
   for exmp,tag in zip(data.sentences(),preds) :
      sid = exmp['sid']
      e1 = exmp['e1']
      e2 = exmp['e2']
      if tag!='null' :
         print(sid, e1, e2, tag, sep="|", file=outf)

   outf.close()

## 9. Evaluation function

In [ ]:
## --------- Evaluator -----------
def evaluation(datadir,outfile) :
   evaluator.evaluate("DDI", datadir, outfile)


In [ ]:
# Validation data
X = codes.encode_words(valdata, features=FEATURES)
model.eval()
with torch.no_grad():
    inputs = {
        'input_ids': torch.tensor(X[0], dtype=torch.long).to(device),
        'attention_mask': torch.tensor(X[1], dtype=torch.long).to(device)
    }
    outputs = model(**inputs)
    preds = outputs.logits.argmax(dim=-1).cpu().numpy()

Y = [codes.idx2label(int(i)) for i in preds]

# extract entities
output_interactions(valdata, Y, outfile)

# evaluate
evaluation(validationdir,outfile)

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.98 GiB. GPU 0 has a total capacity of 14.56 GiB of which 729.81 MiB is free. Including non-PyTorch memory, this process has 13.85 GiB memory in use. Of the allocated memory 11.68 GiB is allocated by PyTorch, and 2.03 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Test data
testdata = Dataset(testfile)

X = codes.encode_words(testdata, features=FEATURES)
model.eval()
with torch.no_grad():
    inputs = {
        'input_ids': torch.tensor(X[0], dtype=torch.long).to(device),
        'attention_mask': torch.tensor(X[1], dtype=torch.long).to(device)
    }
    outputs = model(**inputs)
    preds = outputs.logits.argmax(dim=-1).cpu().numpy()

Y = [codes.idx2label(int(i)) for i in preds]

# extract entities
output_interactions(testdata, Y, outfile)

# evaluate
evaluation(testdir,outfile)

## 10. Evaluation for OOM


In [ ]:
def predict_in_batches(model, X, batch_size=32):
    model.eval()
    all_preds = []

    input_ids = torch.tensor(X[0], dtype=torch.long)
    attention_masks = torch.tensor(X[1], dtype=torch.long)

    pred_dataset = TensorDataset(input_ids, attention_masks)
    pred_loader = DataLoader(pred_dataset, batch_size=batch_size)

    with torch.no_grad():
        for batch in tqdm(pred_loader, desc="Predicting"):
            input_ids_batch, attention_mask_batch = [t.to(device) for t in batch]

            outputs = model(
                input_ids=input_ids_batch,
                attention_mask=attention_mask_batch
            )

            preds = outputs.logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())

    return np.array(all_preds)

In [ ]:
X = codes.encode_words(valdata, features=FEATURES)

preds = predict_in_batches(model, X, batch_size=32)

Y = [codes.idx2label(int(i)) for i in preds]

output_interactions(valdata, Y, outfile)
evaluation(validationdir, outfile)

Predicting:   0%|          | 0/145 [00:00<?, ?it/s]

                   tp	  fp	  fn	#pred	#exp	P	R	F1
------------------------------------------------------------------------------
advise            122	  40	  19	 162	 141	75.3%	86.5%	80.5%
effect            244	  48	  68	 292	 312	83.6%	78.2%	80.8%
int                22	   3	   6	  25	  28	88.0%	78.6%	83.0%
mechanism         188	  50	  73	 238	 261	79.0%	72.0%	75.4%
------------------------------------------------------------------------------
M.avg            -	-	-	-	-	81.5%	78.8%	79.9%
------------------------------------------------------------------------------
m.avg             576	 141	 166	 717	 742	80.3%	77.6%	79.0%
m.avg(no class)   608	 109	 134	 717	 742	84.8%	81.9%	83.3%


In [ ]:
testdata = Dataset(testfile)

X = codes.encode_words(testdata, features=FEATURES)

preds = predict_in_batches(model, X, batch_size=32)

Y = [codes.idx2label(int(i)) for i in preds]

output_interactions(testdata, Y, outfile)
evaluation(testdir, outfile)

Predicting:   0%|          | 0/180 [00:00<?, ?it/s]

                   tp	  fp	  fn	#pred	#exp	P	R	F1
------------------------------------------------------------------------------
advise            161	  42	  48	 203	 209	79.3%	77.0%	78.2%
effect            214	  54	  72	 268	 286	79.9%	74.8%	77.3%
int                20	  10	   5	  30	  25	66.7%	80.0%	72.7%
mechanism         260	  99	  80	 359	 340	72.4%	76.5%	74.4%
------------------------------------------------------------------------------
M.avg            -	-	-	-	-	74.6%	77.1%	75.6%
------------------------------------------------------------------------------
m.avg             655	 205	 205	 860	 860	76.2%	76.2%	76.2%
m.avg(no class)   704	 156	 156	 860	 860	81.9%	81.9%	81.9%
